This notebook performs the Bronze ingestion step for the Automated Data Guard project.

What this notebook does:
* Downloads the raw Kaggle dataset using the Kaggle API
* Uploads the raw CSV file to ADLS Gen2 Bronze
* Verifies the uploaded file metadata

Architecture position:
* Source: Kaggle API
* Orchestration: Azure Data Factory triggers this notebook
* Target: ADLS Gen2 Bronze raw layer
* Downstream: a later notebook will read the Bronze file for data quality checks and split records into Silver and Quarantine

Why Bronze stays raw:
* No cleansing or business transformation is done here
* The goal is to preserve the source data exactly as received for traceability and reprocessing

Execution order:
* Cell 1 installs required libraries
* Cell 2 restarts Python
* Cell 3 loads configuration and secrets
* Cell 4 authenticates to Kaggle and downloads the dataset
* Cell 5 uploads the raw file to ADLS Bronze
* Cell 6 validates that the upload succeeded

Important note:
* Large Bronze files may not open in the Azure portal editor because of UI preview size limits. Validate the load using notebook output and ADLS metadata instead.

Inputs used by this notebook:
* Kaggle username
* Kaggle API token from Databricks secret scope
* ADLS storage account name
* ADLS storage key from Databricks secret scope
* Kaggle dataset identifier and expected CSV file name

Outputs produced by this notebook:
* Raw CSV stored in the `bronze` container under a date-based ingestion folder
* Final ADLS Bronze path printed for downstream consumption
* Uploaded file size and last modified timestamp for verification

Operational behavior:
* The dataset is downloaded to temporary local storage first
* The file is then uploaded to ADLS Bronze as the raw landing object
* This notebook is designed for batch execution and can be called from ADF on a schedule or on demand.

What happens after this notebook:
* ADF triggers the data quality notebook
* The Bronze file is read and validated with business and quality rules
* Passing records move to Silver
* Failing records move to Quarantine and are logged to Azure SQL
* If the failure percentage breaches the threshold, Logic Apps sends an alert

In [0]:
%pip install kaggle azure-storage-file-datalake

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import json
import os
import shutil
import subprocess
from datetime import datetime
from azure.storage.filedatalake import DataLakeServiceClient

kaggle_username = "mounikavanimanda"
kaggle_dataset = "sobhanmoosavi/us-accidents"
kaggle_file_name = "US_Accidents_March23.csv"

storage_account_name = "stdqprojectmouni"
container_name = "bronze"
bronze_folder = f"kaggle/us_accidents/ingestion_date={datetime.utcnow().strftime('%Y-%m-%d')}"
adls_file_path = f"{bronze_folder}/{kaggle_file_name}"

local_download_dir = "/tmp/kaggle_us_accidents"
local_csv_path = f"{local_download_dir}/{kaggle_file_name}"

kaggle_token = dbutils.secrets.get(scope="dq-secret-scope", key="kaggle-api-token")
storage_key = dbutils.secrets.get(scope="dq-secret-scope", key="storage-account-key")

print("Kaggle and ADLS configuration loaded.")
print(f"Bronze target: abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{adls_file_path}")

/home/spark-0a28eac0-42a8-4259-95be-e0/.ipykernel/29944/command-8124137907895878-4043392185:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  bronze_folder = f"kaggle/us_accidents/ingestion_date={datetime.utcnow().strftime('%Y-%m-%d')}"


Kaggle and ADLS configuration loaded.
Bronze target: abfss://bronze@stdqprojectmouni.dfs.core.windows.net/kaggle/us_accidents/ingestion_date=2026-05-24/US_Accidents_March23.csv


In [0]:
os.environ["KAGGLE_CONFIG_DIR"] = "/tmp"
os.makedirs(local_download_dir, exist_ok=True)

with open("/tmp/kaggle.json", "w") as handle:
    json.dump({"username": kaggle_username, "key": kaggle_token}, handle)

os.chmod("/tmp/kaggle.json", 0o600)

if os.path.exists(local_csv_path):
    os.remove(local_csv_path)

archive_path = f"{local_download_dir}/us-accidents.zip"
if os.path.exists(archive_path):
    os.remove(archive_path)

subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        kaggle_dataset,
        "-p",
        local_download_dir,
        "--force",
        "--unzip",
    ],
    check=True,
)

if not os.path.exists(local_csv_path):
    raise FileNotFoundError(f"Expected file not found after download: {local_csv_path}")

print(f"Downloaded file size: {os.path.getsize(local_csv_path) / (1024 * 1024):.2f} MB")

Dataset URL: https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents
License(s): CC-BY-NC-SA-4.0


100%|██████████| 653M/653M [00:26<00:00, 26.1MB/s]



Downloaded file size: 2916.51 MB


In [0]:
service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=storage_key,
)
file_system_client = service_client.get_file_system_client(container_name)
directory_client = file_system_client.get_directory_client(bronze_folder)
directory_client.create_directory()
file_client = file_system_client.get_file_client(adls_file_path)

with open(local_csv_path, "rb") as data:
    file_client.upload_data(data, overwrite=True)

print("Upload to ADLS Bronze completed.")
print(f"ADLS Bronze file: https://{storage_account_name}.dfs.core.windows.net/{container_name}/{adls_file_path}")

Upload to ADLS Bronze completed.
ADLS Bronze file: https://stdqprojectmouni.dfs.core.windows.net/bronze/kaggle/us_accidents/ingestion_date=2026-05-24/US_Accidents_March23.csv


In [0]:
file_properties = file_client.get_file_properties()
print(f"Uploaded bytes: {file_properties.size:,}")
print(f"Last modified: {file_properties.last_modified}")
print(f"Bronze landing path: abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{adls_file_path}")

Uploaded bytes: 3,058,183,727
Last modified: 2026-05-24 10:52:49+00:00
Bronze landing path: abfss://bronze@stdqprojectmouni.dfs.core.windows.net/kaggle/us_accidents/ingestion_date=2026-05-24/US_Accidents_March23.csv


In [0]:
 download = file_client.download_file(offset=0, length=1024 * 1024)
sample_bytes = download.readall()
sample_text = sample_bytes.decode("utf-8", errors="replace")
sample_lines = sample_text.splitlines()

print("First 5 lines from the Bronze CSV:")
for line in sample_lines[:5]:
    print(line)

header_columns = sample_lines[0].split(",") if sample_lines else []
print(f"\nApproximate column count from header: {len(header_columns)}")

First 5 lines from the Bronze CSV:
ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,,,0.01,Right lane blocked due to accident on I-70 Eastbound at Exit 41 OH-235 State Route 4.,I-70 E,Dayton,Montgomery,OH,45424,US,US/Eastern,KFFO,2016-02-08 05:58:00,36.9,,91.0,29.68,10.0,Calm,,0.02,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Night
A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.9280590000

In [0]:
from azure.storage.filedatalake import DataLakeServiceClient
from pyspark.sql import functions as F

storage_account_name = "stdqprojectmouni"
container_name = "bronze"
adls_file_path = "kaggle/us_accidents/ingestion_date=2026-05-24/US_Accidents_March23.csv"
storage_key = dbutils.secrets.get(scope="dq-secret-scope", key="storage-account-key")

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=storage_key,
)
file_system_client = service_client.get_file_system_client(container_name)
file_client = file_system_client.get_file_client(adls_file_path)
file_properties = file_client.get_file_properties()

sample_size_bytes = 8 * 1024 * 1024
sample_bytes = file_client.download_file(offset=0, length=sample_size_bytes).readall()
sample_text = sample_bytes.decode("utf-8", errors="replace")
sample_lines = sample_text.splitlines()

header_columns = sample_lines[0].split(",") if sample_lines else []
data_lines = sample_lines[1:]
rows_in_sample = len(data_lines)
line_count_text = file_client.download_file().readall().decode("utf-8", errors="replace").count("\n")
approx_total_rows = max(line_count_text - 1, 0)

sample_df = spark.createDataFrame([(line,) for line in data_lines[:5]], ["raw_csv_line"])

missing_counts = []
for index, column_name in enumerate(header_columns):
    missing_count = 0
    for raw_line in data_lines:
        values = raw_line.split(",")
        if index >= len(values) or values[index] == "":
            missing_count += 1
    missing_pct = round((missing_count * 100.0 / rows_in_sample), 2) if rows_in_sample else 0.0
    missing_counts.append((column_name, missing_count, missing_pct))

missing_long_df = spark.createDataFrame(missing_counts, ["column_name", "missing_count_in_sample", "missing_pct_in_sample"])

print(f"Dataset size in ADLS: {file_properties.size / (1024 * 1024 * 1024):.2f} GB")
print(f"Approximate total rows from full file line count: {approx_total_rows:,}")
print(f"Number of features: {len(header_columns)}")
print(f"Rows profiled for missing values sample: {rows_in_sample:,}")
print(f"Columns with at least one missing value in sample: {missing_long_df.filter(F.col('missing_count_in_sample') > 0).count()}")

display(missing_long_df.orderBy(F.col("missing_count_in_sample").desc(), F.col("column_name")).limit(15))
display(sample_df)

Dataset size in ADLS: 2.85 GB
Approximate total rows from full file line count: 7,728,394
Number of features: 46
Rows profiled for missing values sample: 22,432
Columns with at least one missing value in sample: 35


column_name,missing_count_in_sample,missing_pct_in_sample
End_Lat,22432,100.0
End_Lng,22432,100.0
Wind_Chill(F),20201,90.05
Precipitation(in),20002,89.17
Wind_Speed(mph),4714,21.01
Humidity(%),351,1.56
Weather_Condition,247,1.1
Visibility(mi),204,0.91
Temperature(F),193,0.86
Pressure(in),107,0.48


raw_csv_line
"A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,,,0.01,Right lane blocked due to accident on I-70 Eastbound at Exit 41 OH-235 State Route 4.,I-70 E,Dayton,Montgomery,OH,45424,US,US/Eastern,KFFO,2016-02-08 05:58:00,36.9,,91.0,29.68,10.0,Calm,,0.02,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Night"
"A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.92805900000001,-82.831184,,,0.01,Accident on Brice Rd at Tussing Rd. Expect delays.,Brice Rd,Reynoldsburg,Franklin,OH,43068-3402,US,US/Eastern,KCMH,2016-02-08 05:51:00,37.9,,100.0,29.65,10.0,Calm,,0.0,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Day"
"A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,,,0.01,Accident on OH-32 State Route 32 Westbound at Dela Palma Rd. Expect delays.,State Route 32,Williamsburg,Clermont,OH,45176,US,US/Eastern,KI69,2016-02-08 06:56:00,36.0,33.3,100.0,29.67,10.0,SW,3.5,,Overcast,False,False,False,False,False,False,False,False,False,False,False,True,False,Night,Night,Day,Day"
"A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.20558199999998,,,0.01,Accident on I-75 Southbound at Exits 52 52B US-35. Expect delays.,I-75 S,Dayton,Montgomery,OH,45417,US,US/Eastern,KDAY,2016-02-08 07:38:00,35.1,31.0,96.0,29.64,9.0,SW,4.6,,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Day,Day,Day"
"A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,,,0.01,Accident on McEwen Rd at OH-725 Miamisburg Centerville Rd. Expect delays.,Miamisburg Centerville Rd,Dayton,Montgomery,OH,45459,US,US/Eastern,KMGY,2016-02-08 07:53:00,36.0,33.3,89.0,29.65,6.0,SW,3.5,,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,True,False,Day,Day,Day,Day"


In [0]:
import io
import pandas as pd
from azure.storage.filedatalake import DataLakeServiceClient

storage_account_name = "stdqprojectmouni"
container_name = "bronze"
adls_file_path = "kaggle/us_accidents/ingestion_date=2026-05-24/US_Accidents_March23.csv"
storage_key = dbutils.secrets.get(scope="dq-secret-scope", key="storage-account-key")

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=storage_key,
)
file_client = service_client.get_file_system_client(container_name).get_file_client(adls_file_path)

sample_bytes = file_client.download_file(offset=0, length=2 * 1024 * 1024).readall()
sample_text = sample_bytes.decode("utf-8", errors="replace")
complete_text = sample_text.rsplit("\n", 1)[0]
first_ten_df = pd.read_csv(io.StringIO(complete_text), nrows=10)

display(first_ten_df)

ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,null,null,0.01,Right lane blocked due to accident on I-70 Eastbound at Exit 41 OH-235 State Route 4.,I-70 E,Dayton,Montgomery,OH,45424,US,US/Eastern,KFFO,2016-02-08 05:58:00,36.9,null,91.0,29.68,10.0,Calm,null,0.02,Light Rain,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Night,Night,Night
A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.92805900000001,-82.831184,null,null,0.01,Accident on Brice Rd at Tussing Rd. Expect delays.,Brice Rd,Reynoldsburg,Franklin,OH,43068-3402,US,US/Eastern,KCMH,2016-02-08 05:51:00,37.9,null,100.0,29.65,10.0,Calm,null,0.0,Light Rain,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Night,Night,Day
A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,null,null,0.01,Accident on OH-32 State Route 32 Westbound at Dela Palma Rd. Expect delays.,State Route 32,Williamsburg,Clermont,OH,45176,US,US/Eastern,KI69,2016-02-08 06:56:00,36.0,33.3,100.0,29.67,10.0,SW,3.5,null,Overcast,false,false,false,false,false,false,false,false,false,false,false,true,false,Night,Night,Day,Day
A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.20558199999998,null,null,0.01,Accident on I-75 Southbound at Exits 52 52B US-35. Expect delays.,I-75 S,Dayton,Montgomery,OH,45417,US,US/Eastern,KDAY,2016-02-08 07:38:00,35.1,31.0,96.0,29.64,9.0,SW,4.6,null,Mostly Cloudy,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Day,Day,Day
A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,null,null,0.01,Accident on McEwen Rd at OH-725 Miamisburg Centerville Rd. Expect delays.,Miamisburg Centerville Rd,Dayton,Montgomery,OH,45459,US,US/Eastern,KMGY,2016-02-08 07:53:00,36.0,33.3,89.0,29.65,6.0,SW,3.5,null,Mostly Cloudy,false,false,false,false,false,false,false,false,false,false,false,true,false,Day,Day,Day,Day
A-6,Source2,3,2016-02-08 07:44:26,2016-02-08 08:14:26,40.10059,-82.92519399999998,null,null,0.01,Accident on I-270 Outerbelt Northbound near Exit 29 OH-3 State St. Expect delays.,Westerville Rd,Westerville,Franklin,OH,43081,US,US/Eastern,KCMH,2016-02-08 07:51:00,37.9,35.5,97.0,29.63,7.0,SSW,3.5,0.03,Light Rain,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,Day,Day,Day
A-7,Source2,2,2016-02-08 07:59:35,2016-02-08 08:29:35,39.758274,-84.23050699999997,null,null,0.0,Accident on Oakridge Dr at Woodward Ave. Expect delays.,N Woodward Ave,Dayton,Montgomery,OH,45417-2476,US,US/Eastern,KDAY,2016-02-08 07:56:00,34.0,31.0,100.0,29.66,7.0,WSW,3.5,null,Overcast,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,Day,Day,Day
A-8,Source2,3,2016-02-08 07:59:58,2016-02-08 08:29:58,39.770382,-84.194901,null,null,0.01,Accident on I-75 Southbound at Exit 54B Grand Ave. Expect delays.,N Main St,Dayton,Montgomery,OH,45405,US,US/Eastern,KDAY,2016-02-08 07:56:00,34.0,31.0,100.0,29.66,7.0,WSW,3.5,null,Overcast,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,Day,Day,Day
A-9,Source2,2,2016-02-08 08:00:40,2016-02-08 08:30:40,39.778061,-84.172005,null,null,0.0,Accident on Notre Dame Ave at Warner Ave. Expect delays.,Notre Dame Ave,Dayton,Montgomery,OH,45404-1923,US,US/Eastern,KFFO,2016-02-08 07:58:00,33.3,null,99.0,29.67,5.0,SW,1.2,null,Mostly Cloudy,false,false,false,false,false,false,false,false,false,false,false,false,false